## Feature Selection: Cross-Validated SelectKBest (ANOVA F-test)

RNA-Seq data is extremely high-dimensional and contains substantial redundancy.
To reduce dimensionality while preserving discriminative information, we apply
explicit feature selection using SelectKBest with an ANOVA F-test.

The number of selected genes (K) is treated as a hyperparameter and is chosen
using stratified K-fold cross-validation on the training set to ensure
leakage-safe and data-driven selection.

In [2]:
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression

In [3]:
# Load preprocessed features and labels
X_train_proc = np.load("processed/X_train_proc.npy")
X_test_proc  = np.load("processed/X_test_proc.npy")
y_train = np.load("processed/y_train.npy")
y_test  = np.load("processed/y_test.npy")

# Load label names
label_classes = pd.read_csv("processed/label_classes.csv")["class_name"].tolist()

print("Train shape:", X_train_proc.shape)
print("Test shape:", X_test_proc.shape)
print("Classes:", label_classes)

Train shape: (640, 20254)
Test shape: (161, 20254)
Classes: ['BRCA', 'COAD', 'KIRC', 'LUAD', 'PRAD']


### Step 1 — Define candidate K values and cross-validation strategy

We evaluate multiple values of K to determine how many genes are required
to achieve optimal generalization performance.

In [5]:
K_values = [200, 500, 800, 1000, 1500, 2000, 3000, 5000]

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

### Step 2 — Cross-validation with feature selection inside each fold

Feature selection is performed inside the cross-validation loop to avoid
data leakage. A Logistic Regression classifier is used to evaluate the
predictive utility of each selected feature subset.

In [7]:
cv_results = []

for K in K_values:
    pipeline = Pipeline([
        ("select", SelectKBest(score_func=f_classif, k=K)),
        ("clf", LogisticRegression(
            max_iter=3000,
            solver="lbfgs",
            multi_class="multinomial",
            random_state=42
        ))
    ])
    
    scores = cross_val_score(
        pipeline,
        X_train_proc,
        y_train,
        cv=cv,
        scoring="f1_macro",
        n_jobs=-1
    )
    
    cv_results.append({
        "K": K,
        "mean_cv_accuracy": scores.mean(),
        "std_cv_accuracy": scores.std()
    })

cv_results_df = pd.DataFrame(cv_results).sort_values(
    "mean_cv_accuracy",
    ascending=False
)

cv_results_df

,K,mean_cv_accuracy,std_cv_accuracy
7,5000,0.998728,0.002544
1,500,0.998657,0.002685
2,800,0.998657,0.002685
3,1000,0.998657,0.002685
4,1500,0.998657,0.002685
5,2000,0.998657,0.002685
6,3000,0.998657,0.002685
0,200,0.997397,0.003187


**Inference:**  
Cross-validation shows that classification performance saturates after a few
hundred genes, indicating that only a compact subset of genes is required
to capture most discriminative informatin.


### Step 3 — Select optimal K

We select the value of K that achieves the highest mean cross-validated accuracy.
If multiple values lie within a performance plateau, the smaller K is preferred
to reduce model complexity.

In [10]:
best_row = cv_results_df.iloc[0]
best_K = int(best_row["K"])

print("Selected K:", best_K)
print("Mean CV accuracy:", best_row["mean_cv_accuracy"])
print("CV standard deviation:", best_row["std_cv_accuracy"])

Selected K: 5000
Mean CV accuracy: 0.9987278835386337
CV standard deviation: 0.002544232922732359


**Inference:**  
The selected value of K provides the best trade-off between predictive
performance and dimensionality reduction.

### Step 4 — Fit final selector on full training set

After selecting K, SelectKBest is fitted on the full training set and
applied to both training and test data to obtain the final feature matrics.


In [13]:
final_selector = SelectKBest(score_func=f_classif, k=best_K)

X_train_fs = final_selector.fit_transform(X_train_proc, y_train)
X_test_fs = final_selector.transform(X_test_proc)

print("Original features:", X_train_proc.shape[1])
print("Selected features:", X_train_fs.shape[1])
print("Train shape:", X_train_fs.shape)
print("Test shape:", X_test_fs.shape)

Original features: 20254
Selected features: 5000
Train shape: (640, 5000)
Test shape: (161, 5000)


**Inference:**  
The dataset is now represented by a reduced set of highly discriminative genes,
improving computational efficiency and supporting downstream interpretation.

### Step 5 — Retrieve selected gene names

Extracting selected gene identifiers enables biological interpretation and
feature importance analysis in later stages.

In [16]:
kept_genes = pd.read_csv("processed/kept_genes.csv")["gene"].to_numpy()
selected_genes = kept_genes[final_selector.get_support()]

print("Number of selected genes:", len(selected_genes))
print("First 15 selected genes:", selected_genes[:15])

Number of selected genes: 5000
First 15 selected genes: ['gene_18' 'gene_26' 'gene_28' 'gene_29' 'gene_30' 'gene_34' 'gene_35'
 'gene_36' 'gene_39' 'gene_44' 'gene_45' 'gene_46' 'gene_47' 'gene_55'
 'gene_61']


### Step 6 — Final feature matrices for modeling

From this point onward, all supervised models are trained using the
feature-selected matrices.

In [18]:
X_train_final = X_train_fs
X_test_final = X_test_fs

print("Final train shape:", X_train_final.shape)
print("Final test shape:", X_test_final.shape)

Final train shape: (640, 5000)
Final test shape: (161, 5000)


**Inference:**  
Logistic Regression achieves meaningful classification performance,
confirming that RNA-Seq gene expression profiles contain strong predictive
signal. Misclassifications are mainly observed between biologically related
cancer types, consistent with overlap seen during DA.
